# Geometry Statistics for ViT Normalization

Computes empirical `mu` and `sigma` for the two geometry scalars used in the Early-Fusion ViT:
- **X_area** = `(log(A + 1) - mu_a) / sigma_a` — normalized component pixel area
- **X_size** = `(log(L + 1) - mu_s) / sigma_s` — normalized max bounding box dimension

**Data source**: `real` + `synth` splits of `Incinciblecolonel/MathStrike` on HuggingFace.
These splits directly provide `box_area` and `comp_size` columns, so no local tokenization is needed.
This gives accurate stats over thousands of samples (vs. the previous 10-sample estimate from `tokenization_results`).

In [7]:
!pip install datasets

   ---------------------------------------- 0.0/527.0 kB ? eta -:--:--
   ---------------------------------------- 527.0/527.0 kB 16.6 MB/s  0:00:00

   ------------- -------------------------- 1/3 [multiprocess]
   -------------------------- ------------- 2/3 [datasets]
   -------------------------- ------------- 2/3 [datasets]
   -------------------------- ------------- 2/3 [datasets]
   ---------------------------------------- 3/3 [datasets]



In [1]:
import os
import math
import numpy as np
from datasets import load_dataset
from dotenv import load_dotenv
from tqdm import tqdm

load_dotenv()
hf_token = os.getenv('HF_token')

c:\Users\Urmi Kanrar\anaconda3\envs\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ── Configure how many samples to use ────────────────────────────────────
# Set to None to use the entire dataset (will take longer)
SAMPLE_LIMIT = None

areas = []
sizes = []

for data_dir in ['real', 'synth']:
    for split in ['train', 'test']:
        try:
            ds = load_dataset(
                'Incinciblecolonel/MathStrike',
                data_dir=data_dir,
                split=split,
                streaming=True,
                token=hf_token
            )
            count = 0
            for sample in tqdm(ds, desc=f'{data_dir}/{split}'):
                A = sample['box_area']
                comp_size = sample['comp_size']     # [height, width]
                L = max(comp_size[0], comp_size[1])

                areas.append(math.log(A + 1))
                sizes.append(math.log(L + 1))

                count += 1
                if SAMPLE_LIMIT and count >= SAMPLE_LIMIT:
                    break
        except Exception as e:
            print(f'Skipping {data_dir}/{split}: {e}')

print(f'\nTotal components collected: {len(areas)}')

real/train: 97223it [01:11, 1365.01it/s]
real/test: 1000it [00:05, 194.34it/s]
synth/train: 97223it [01:21, 1189.47it/s]
synth/test: 1000it [00:03, 287.14it/s]


Total components collected: 196446


In [3]:
# ── Compute final statistics ──────────────────────────────────────────────
mu_a    = np.mean(areas)
sigma_a = np.std(areas)
mu_s    = np.mean(sizes)
sigma_s = np.std(sizes)

print('=' * 50)
print(f'Computed over {len(areas):,} components')
print('=' * 50)
print(f'Area Mean   (mu_a):    {mu_a:.4f}')
print(f'Area Std    (sigma_a): {sigma_a:.4f}')
print()
print(f'Size Mean   (mu_s):    {mu_s:.4f}')
print(f'Size Std    (sigma_s): {sigma_s:.4f}')
print('=' * 50)
print()
print('Copy these into vit_strikeout_detector.py and train_vit.py:')
print(f'MU_A    = {mu_a:.4f}')
print(f'SIGMA_A = {sigma_a:.4f}')
print(f'MU_S    = {mu_s:.4f}')
print(f'SIGMA_S = {sigma_s:.4f}')

Computed over 196,446 components
Area Mean   (mu_a):    8.6510
Area Std    (sigma_a): 0.9201

Size Mean   (mu_s):    5.1047
Size Std    (sigma_s): 0.6324

Copy these into vit_strikeout_detector.py and train_vit.py:
MU_A    = 8.6510
SIGMA_A = 0.9201
MU_S    = 5.1047
SIGMA_S = 0.6324
